# Apriori Algorithm Implementation Assignment

### Objective:
You will implement the **Apriori algorithm** from scratch (i.e., without using any libraries like `mlxtend`) to find frequent itemsets and generate association rules.

### Dataset:
Use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle. You can filter it for a specific country (e.g., `United Kingdom`) and time range to reduce size if needed.

---

## Step 1: Data Preprocessing

- Load the dataset
- Remove rows with missing values
- Filter out rows where `Quantity <= 0`
- Convert Data into Basket Format

👉 **Implement code below**

In [12]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np

# --- Step 2: Load Dataset ---
# Make sure you have uploaded "Online Retail.xlsx"
data = pd.read_csv('OnlineRetail.csv',encoding = 'ISO-8859-1')

# --- Step 3: Data Cleaning ---
data.dropna(inplace=True)                                       # remove nulls
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')] # remove cancelled invoices
data = data[data['Quantity'] > 0]                               # keep only positive quantity

In [14]:

# --- Step 4: Prepare Transaction Dataset ---
# Group transactions by InvoiceNo into list of items
transactions = (data.groupby("InvoiceNo")['Description']
                   .apply(list)
                   .tolist())

print("Sample Transaction:", transactions[0][:10])  # show first 10 items of first invoice

Sample Transaction: ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER']


## Step 2: Implement Apriori Algorithm
Step-by-Step Procedure:
1. Generate Frequent 1-Itemsets
Count the frequency (support) of each individual item in the dataset.
Keep only those with support ≥ min_support.
→ Result is L1 (frequent 1-itemsets)
2. Iterative Candidate Generation (k = 2 to n)
While L(k-1) is not empty:
a. Candidate Generation

Generate candidate itemsets Ck of size k from L(k-1) using the Apriori property:
Any (k-itemset) is only frequent if all of its (k−1)-subsets are frequent.
b. Prune Candidates
Eliminate candidates that have any (k−1)-subset not in L(k-1).
c. Count Support
For each transaction, count how many times each candidate in Ck appears.
d. Generate Frequent Itemsets
Form Lk by keeping candidates from Ck that meet the min_support.
Repeat until Lk becomes empty.
Implement the following functions:
1. `get_frequent_itemsets(transactions, min_support)` - Returns frequent itemsets and their support
2. `generate_candidates(prev_frequent_itemsets, k)` - Generates candidate itemsets of length `k`
3. `calculate_support(transactions, candidates)` - Calculates the support count for each candidate

**Write reusable functions** for each part of the algorithm.

In [16]:
# --- Step 5: Support Calculation Function ---
def get_support(itemset, transactions):
    """
    Calculate support of an itemset
    """
    count = 0
    for trans in transactions:
        if set(itemset).issubset(set(trans)):
            count += 1
    return count / len(transactions)


In [18]:
# --- Step 6: Candidate Generation ---
def generate_candidates(prev_frequent_itemsets, k):
    """
    Generate candidate itemsets of size k from frequent itemsets of size (k-1).
    """
    candidates = []
    prev_items = list(prev_frequent_itemsets.keys())

    for i in range(len(prev_items)):
        for j in range(i+1, len(prev_items)):
            # Union of two sets
            union_set = tuple(sorted(set(prev_items[i]) | set(prev_items[j])))
            if len(union_set) == k and union_set not in candidates:
                candidates.append(union_set)
    return candidates

In [20]:




# --- Step 7: Apriori Algorithm ---
def apriori(transactions, min_support=0.01):
    """
    Apriori algorithm implemented from scratch.
    Returns dict of frequent itemsets with their support.
    """
    item_support = {}
    num_trans = len(transactions)

    # Step 1: Get frequent 1-itemsets
    items = set(i for trans in transactions for i in trans)
    freq_itemsets = {}
    
    for item in items:
        sup = get_support([item], transactions)
        if sup >= min_support:
            freq_itemsets[(item,)] = sup

    all_freq_itemsets = dict(freq_itemsets)

    k = 2
    while freq_itemsets:
        # Step 2: Generate candidates of size k
        candidates = generate_candidates(freq_itemsets, k)

        freq_itemsets = {}
        for cand in candidates:
            sup = get_support(cand, transactions)
            if sup >= min_support:
                freq_itemsets[cand] = sup

        all_freq_itemsets.update(freq_itemsets)
        k += 1

    return all_freq_itemsets


In [22]:
# --- Step 8: Run Apriori ---
frequent_itemsets = apriori(transactions, min_support=0.02)

print("\nTop Frequent Itemsets:")
for item, sup in sorted(frequent_itemsets.items(), key=lambda x: -x[1])[:10]:
    print(item, ":", round(sup, 4))



Top Frequent Itemsets:
('WHITE HANGING HEART T-LIGHT HOLDER',) : 0.1063
('REGENCY CAKESTAND 3 TIER',) : 0.0919
('JUMBO BAG RED RETROSPOT',) : 0.0863
('PARTY BUNTING',) : 0.0744
('ASSORTED COLOUR BIRD ORNAMENT',) : 0.0742
('LUNCH BAG RED RETROSPOT',) : 0.0695
('SET OF 3 CAKE TINS PANTRY DESIGN ',) : 0.0618
('POSTAGE',) : 0.0593
('LUNCH BAG  BLACK SKULL.',) : 0.0568
('PACK OF 72 RETROSPOT CAKE CASES',) : 0.0555


## Step 3: Generate Association Rules

- Use frequent itemsets to generate association rules
- For each rule `A => B`, calculate:
  - **Support**
  - **Confidence**
- Only return rules that meet a minimum confidence threshold (e.g., 0.5)

👉 **Implement rule generation function below**

In [24]:
# Function to generate rules from frequent itemsets

def generate_rules(frequent_itemsets, min_confidence):
    rules = []

    for itemset, support in frequent_itemsets.items():
        if len(itemset) > 1:
            # split into all possible A → B
            n = len(itemset)
            for i in range(n):
                for j in range(i+1, n):
                    A = tuple([itemset[i]])
                    B = tuple(sorted(set(itemset) - set(A)))
                    
                    support_AB = support
                    support_A = frequent_itemsets.get(A, 0)
                    if support_A > 0:
                        confidence = support_AB / support_A
                        if confidence >= min_confidence:
                            rules.append((A, B, support_AB, confidence))
    return sorted(rules, key=lambda x: x[3], reverse=True)


## Step 4: Output and Visualize

- Print top 10 frequent itemsets
- Print top 10 association rules (by confidence or lift)

👉 **Output results below**

In [27]:
# Output the final results
# Optional: Add visualizations
# Run Apriori
# --- Step 9: Association Rule Generation ---
def generate_rules(frequent_itemsets, transactions, min_conf=0.5):
    """
    Generate association rules from frequent itemsets.
    """
    rules = []
    for itemset, sup in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for i in range(len(itemset)):
            antecedent = (itemset[i],)
            consequent = tuple(sorted(set(itemset) - set(antecedent)))

            support_ab = sup
            support_a = get_support(antecedent, transactions)
            support_b = get_support(consequent, transactions)

            confidence = support_ab / support_a if support_a > 0 else 0
            lift = confidence / support_b if support_b > 0 else 0

            if confidence >= min_conf:
                rules.append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": round(support_ab, 4),
                    "confidence": round(confidence, 4),
                    "lift": round(lift, 4)
                })
    return pd.DataFrame(rules)


rules_df = generate_rules(frequent_itemsets, transactions, min_conf=0.5)

print("\nTop Association Rules:")
print(rules_df.head())

# Your Code Here


Top Association Rules:
                              antecedent  \
0          (ALARM CLOCK BAKELIKE GREEN,)   
1           (ALARM CLOCK BAKELIKE RED ,)   
2  (GARDENERS KNEELING PAD CUP OF TEA ,)   
3   (GARDENERS KNEELING PAD KEEP CALM ,)   
4              (LUNCH BAG SUKI DESIGN ,)   

                              consequent  support  confidence     lift  
0           (ALARM CLOCK BAKELIKE RED ,)   0.0286      0.6717  14.1976  
1          (ALARM CLOCK BAKELIKE GREEN,)   0.0286      0.6043  14.1976  
2   (GARDENERS KNEELING PAD KEEP CALM ,)   0.0250      0.7291  17.8773  
3  (GARDENERS KNEELING PAD CUP OF TEA ,)   0.0250      0.6124  17.8773  
4             (LUNCH BAG RED RETROSPOT,)   0.0243      0.5006   7.2037  


In [ ]:
# import pandas as pd
# from itertools import chain, combinations

# # ----------------------------
# # Sample dataset
# # ----------------------------
# transactions = [
#     [1, 2, 3],
#     [2, 3, 5],
#     [1, 2, 3, 5],
#     [2, 5]
# ]

# # ----------------------------
# # Support calculation
# # ----------------------------
# def get_support(itemset, transactions):
#     """Return support (count/len) of an itemset in the transactions."""
#     count = sum(1 for t in transactions if set(itemset).issubset(set(t)))
#     return count / len(transactions), count

# # ----------------------------
# # Apriori frequent itemsets
# # ----------------------------
# def apriori(transactions, min_sup=0.5):
#     items = sorted({i for t in transactions for i in t})
#     freq_itemsets = {}

#     # singletons
#     for item in items:
#         sup, cnt = get_support([item], transactions)
#         if sup >= min_sup:
#             freq_itemsets[(item,)] = sup

#     k = 2
#     while True:
#         candidates = [tuple(sorted(set(a) | set(b)))
#                       for a in freq_itemsets
#                       for b in freq_itemsets
#                       if len(set(a) | set(b)) == k]
#         candidates = sorted(set(candidates))

#         new_sets = {}
#         for c in candidates:
#             sup, _ = get_support(c, transactions)
#             if sup >= min_sup:
#                 new_sets[c] = sup
#         if not new_sets:
#             break
#         freq_itemsets.update(new_sets)
#         k += 1
#     return freq_itemsets

# # ----------------------------
# # Generate rules
# # ----------------------------
# def all_subsets(itemset):
#     return chain.from_iterable(combinations(itemset, r) for r in range(1, len(itemset)))

# def generate_rules(freq_itemsets, transactions):
#     rules = []
#     for itemset, sup in freq_itemsets.items():
#         if len(itemset) < 2:
#             continue

#         _, sup_count_itemset = get_support(itemset, transactions)

#         # generate ALL non-empty proper subsets
#         for r in range(1, len(itemset)):
#             for antecedent in combinations(itemset, r):
#                 consequent = tuple(sorted(set(itemset) - set(antecedent)))

#                 sup_a, sup_count_a = get_support(antecedent, transactions)
#                 if sup_count_a == 0:
#                     continue

#                 confidence = sup_count_itemset / sup_count_a
#                 confidence_pct = f"{round(confidence * 100)} %"
#                 conf_fraction = f"{sup_count_itemset}/{sup_count_a} = {round(confidence,2)}"

#                 rules.append({
#                     "Association Rule": f"{' ^ '.join(map(str, antecedent))} → {' ^ '.join(map(str, consequent))}",
#                     "Support": sup_count_itemset,
#                     "Confidence": conf_fraction,
#                     "Confidence (%)": confidence_pct
#                 })
#     return pd.DataFrame(rules)


# # ----------------------------
# # Run Apriori + Rules
# # ----------------------------
# min_support = 0.5
# freq_itemsets = apriori(transactions, min_sup=min_support)
# rules_df = generate_rules(freq_itemsets, transactions)

# print("Frequent Itemsets:")
# print(freq_itemsets)
# print("\nRules Generation:")
# print(rules_df)


In [29]:
from itertools import combinations

# Dataset (transactions)
transactions = {
    100: [1, 3, 4],
    200: [2, 3, 5],
    300: [1, 2, 3, 5],
    400: [2, 5]
}

# Parameters
min_support = 2
min_confidence = 0.0  # we want all rules (filter later if needed)

# Step 1: Count support for single items (C1)
def get_support(itemset, transactions):
    count = 0
    for tid, items in transactions.items():
        if set(itemset).issubset(set(items)):
            count += 1
    return count

# Step 2: Generate candidates
def generate_candidates(prev_frequent, k):
    candidates = []
    items = list(set([i for s in prev_frequent for i in s]))
    items.sort()
    for comb in combinations(items, k):
        candidates.append(comb)
    return candidates

# Step 3: Apriori algorithm
frequent_itemsets = {}
k = 1

# C1 -> L1
candidates = generate_candidates([tuple([i]) for i in set(i for t in transactions.values() for i in t)], k)
L = []
for c in candidates:
    sup = get_support(c, transactions)
    if sup >= min_support:
        L.append((c, sup))
frequent_itemsets[k] = L

# Generate further
while frequent_itemsets[k]:
    k += 1
    prev_frequent = [x[0] for x in frequent_itemsets[k-1]]
    candidates = []
    # Join step
    for comb in combinations(set(i for s in prev_frequent for i in s), k):
        candidates.append(comb)

    L = []
    for c in candidates:
        sup = get_support(c, transactions)
        if sup >= min_support:
            L.append((c, sup))
    frequent_itemsets[k] = L

# Remove last empty
if not frequent_itemsets[k]:
    del frequent_itemsets[k]
    

# Print Frequent Itemsets
print("Frequent Itemsets:")
for k, itemsets in frequent_itemsets.items():
    for itemset, sup in itemsets:
        print(f"{itemset} -> support = {sup}")

# Step 4: Association Rules
print("\nAssociation Rules:")
rules = []
for k, itemsets in frequent_itemsets.items():
    for itemset, sup_count in itemsets:
        if len(itemset) > 2:
            for i in range(1, len(itemset)):
                for lhs in combinations(itemset, i):
                    rhs = tuple(sorted(set(itemset) - set(lhs)))
                    lhs_support = get_support(lhs, transactions)
                    conf = sup_count / lhs_support
                    rules.append((lhs, rhs, sup_count, conf))

# Format Output same as slide
for lhs, rhs, sup, conf in rules:
    lhs_str = "^".join(map(str, lhs))
    rhs_str = "^".join(map(str, rhs))
    print(f"{lhs_str} -> {rhs_str}, Support={sup}, Confidence={sup}/{get_support(lhs, transactions)} = {conf:.2f}, {conf*100:.0f}%")


Frequent Itemsets:
(1,) -> support = 2
(2,) -> support = 3
(3,) -> support = 3
(5,) -> support = 3
(1, 3) -> support = 2
(2, 3) -> support = 2
(2, 5) -> support = 3
(3, 5) -> support = 2
(2, 3, 5) -> support = 2

Association Rules:
2 -> 3^5, Support=2, Confidence=2/3 = 0.67, 67%
3 -> 2^5, Support=2, Confidence=2/3 = 0.67, 67%
5 -> 2^3, Support=2, Confidence=2/3 = 0.67, 67%
2^3 -> 5, Support=2, Confidence=2/2 = 1.00, 100%
2^5 -> 3, Support=2, Confidence=2/3 = 0.67, 67%
3^5 -> 2, Support=2, Confidence=2/2 = 1.00, 100%
